# Regresión Lineal Múltiple — Ventas Walmart

**¿Qué es la Regresión Lineal Múltiple (RLM)?**  
Busca una relación **lineal (línea recta)** entre varias variables de entrada y una variable a predecir.

```
ventas = w1·Store + w2·Holiday + w3·Temperatura + w4·CPI + ... + b
```

Cada `w` es un coeficiente que el modelo aprende — indica cuánto y en qué dirección afecta cada variable a las ventas.

**Dataset:** Walmart.csv — 6,435 registros, 45 tiendas  
**Variable objetivo (Y):** `Weekly_Sales` (ventas semanales en USD)

In [ ]:
# ============================================================
# 1. LIBRERÍAS
# ============================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [ ]:
# ============================================================
# 2. CARGAR Y PREPARAR DATOS
# ============================================================
# La columna Date es texto — no sirve directo como número.
# Extraemos mes y semana del año, que sí son valores numéricos útiles.

df = pd.read_csv("Walmart.csv")
df['Date']  = pd.to_datetime(df['Date'], dayfirst=True)
df['month'] = df['Date'].dt.month
df['week']  = df['Date'].dt.isocalendar().week.astype(int)
df = df.drop(columns=['Date'])

print("Columnas disponibles:", df.columns.tolist())
print(f"Registros: {len(df)} | Tiendas: {df['Store'].nunique()}")
print("\nPrimeras filas:")
print(df.head())

In [ ]:
# ============================================================
# 3. ANÁLISIS EXPLORATORIO
# ============================================================
# Antes de entrenar hay que entender los datos:
# ¿cuánto vende cada tienda? ¿hay estacionalidad por mes?

tabla_tiendas = (df.groupby('Store')['Weekly_Sales']
                   .agg(['mean','std','min','max'])
                   .rename(columns={'mean':'Promedio','std':'Desv.Std',
                                    'min':'Mínimo','max':'Máximo'})
                   .round(0)
                   .sort_values('Promedio', ascending=False))

print("Ventas promedio por tienda (top 10):")
print(tabla_tiendas.head(10).to_string())

t_max = tabla_tiendas['Promedio'].idxmax()
t_min = tabla_tiendas['Promedio'].idxmin()
ratio = tabla_tiendas.loc[t_max,'Promedio'] / tabla_tiendas.loc[t_min,'Promedio']

print(f"\n→ Mayor: Tienda #{t_max} (${tabla_tiendas.loc[t_max,'Promedio']:,.0f})")
print(f"→ Menor: Tienda #{t_min} (${tabla_tiendas.loc[t_min,'Promedio']:,.0f})")
print(f"→ Ratio: {ratio:.1f}x — hay tiendas que venden 8 veces más que otras.")

In [ ]:
# ============================================================
# 4. GRÁFICAS EXPLORATORIAS
# ============================================================

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Ventas promedio por tienda
ventas_ord = tabla_tiendas['Promedio'].sort_values()
axes[0].barh(ventas_ord.index.astype(str), ventas_ord.values,
             color='steelblue', edgecolor='white', height=0.7)
axes[0].axvline(ventas_ord.mean(), color='orangered', linestyle='--',
                linewidth=1.5, label='Promedio')
axes[0].set_title('Ventas promedio por tienda', fontsize=12)
axes[0].set_xlabel('USD/semana')
axes[0].set_ylabel('Tienda')
axes[0].legend(fontsize=9)

# Ventas por mes
ventas_mes = df.groupby('month')['Weekly_Sales'].mean()
meses = ['Ene','Feb','Mar','Abr','May','Jun','Jul','Ago','Sep','Oct','Nov','Dic']
colores = ['tomato' if m in [11,12] else 'steelblue' for m in range(1,13)]
axes[1].bar(range(1,13), ventas_mes.values, color=colores, edgecolor='white')
axes[1].set_xticks(range(1,13))
axes[1].set_xticklabels(meses, rotation=45, fontsize=8)
axes[1].axhline(ventas_mes.mean(), color='gray', linestyle='--', linewidth=1)
axes[1].set_title('Ventas promedio por mes\n(Nov/Dic en rojo = pico navideño)', fontsize=12)
axes[1].set_ylabel('USD')

# Distribución general
axes[2].hist(df['Weekly_Sales']/1e6, bins=50, color='steelblue',
             edgecolor='white', alpha=0.85)
axes[2].axvline(df['Weekly_Sales'].mean()/1e6, color='orangered',
                linestyle='--', linewidth=1.5, label='Media')
axes[2].set_title('Distribución de ventas semanales', fontsize=12)
axes[2].set_xlabel('Millones USD')
axes[2].set_ylabel('Frecuencia')
axes[2].legend(fontsize=9)

plt.suptitle('Análisis Exploratorio — Walmart', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print("→ La gráfica 1 muestra grupos muy distintos entre tiendas (no lineal).")
print("→ La gráfica 2 muestra un pico estacional en Nov/Dic (no lineal).")
print("→ La gráfica 3 muestra una distribución con cola larga (outliers).")

In [ ]:
# ============================================================
# 5. CORRELACIÓN DE PEARSON
# ============================================================
# Mide qué tan relacionada está cada variable con Weekly_Sales.
# r cercano a ±1 = relación fuerte | r cercano a 0 = relación nula.

features = ['Store','Holiday_Flag','Temperature',
            'Fuel_Price','CPI','Unemployment','month','week']

correlacion = (df[features + ['Weekly_Sales']]
               .corr()['Weekly_Sales']
               .drop('Weekly_Sales')
               .sort_values(key=abs, ascending=False))

def nivel(r):
    a = abs(r)
    if a >= 0.6: return 'Alta'
    elif a >= 0.4: return 'Moderada'
    elif a >= 0.2: return 'Baja'
    elif a >= 0.05: return 'Muy baja'
    return 'Nula'

tabla_corr = pd.DataFrame({
    'Variable':       correlacion.index,
    'r':              correlacion.round(4).values,
    'Nivel':          [nivel(v) for v in correlacion.values],
    'Dirección':      ['↑ sube' if v>0.05 else '↓ baja' if v<-0.05
                       else '→ sin efecto claro' for v in correlacion.values]
})
print(tabla_corr.to_string(index=False))

var_top = correlacion.abs().idxmax()
print(f"\n→ Variable más correlacionada: '{var_top}' (r={correlacion[var_top]:.3f})")
print("→ Todas son bajas — ninguna variable sola predice bien las ventas.")

In [ ]:
# ============================================================
# 6. NORMALIZACIÓN MIN-MAX
# ============================================================
# Escala todas las variables al rango [0, 1] para que ninguna
# domine al modelo por tener números más grandes.
# Fórmula: X_norm = (X - X_min) / (X_max - X_min)
# Guardamos min/max para desnormalizar la predicción al final.

X = df[features].values
y = df['Weekly_Sales'].values

X_min, X_max = X.min(axis=0), X.max(axis=0)
y_min, y_max = y.min(), y.max()

X_norm = (X - X_min) / (X_max - X_min)
y_norm = (y - y_min) / (y_max - y_min)

print("Rangos originales de cada variable:")
for nombre, mn, mx in zip(features, X_min, X_max):
    print(f"  {nombre:<15}: {mn:.2f} – {mx:.2f}")
print(f"  {'Weekly_Sales':<15}: ${y_min:,.0f} – ${y_max:,.0f}")
print("\nDespués de normalizar: todos en rango [0.0 – 1.0]")

In [ ]:
# ============================================================
# 7. DIVIDIR EN ENTRENAMIENTO Y PRUEBA (80% / 20%)
# ============================================================
X_train, X_test, y_train, y_test = train_test_split(
    X_norm, y_norm, test_size=0.2, random_state=42
)
print(f"Entrenamiento: {X_train.shape[0]} registros")
print(f"Prueba:         {X_test.shape[0]} registros")

In [ ]:
# ============================================================
# 8. ENTRENAR MODELO
# ============================================================
modelo = LinearRegression()
modelo.fit(X_train, y_train)

tabla_coef = pd.DataFrame({
    'Variable':    features,
    'Coeficiente': modelo.coef_.round(4),
    'Efecto':      ['↑ sube ventas' if c>0 else '↓ baja ventas' for c in modelo.coef_]
}).sort_values('Coeficiente', key=abs, ascending=False)

print(f"Intercepto: {modelo.intercept_:.4f}")
print("\nCoeficientes (mayor valor absoluto = más influencia):")
print(tabla_coef.to_string(index=False))

In [ ]:
# ============================================================
# 9. MÉTRICAS
# ============================================================
predicciones = modelo.predict(X_test)

r2      = r2_score(y_test, predicciones)
mae     = mean_absolute_error(y_test, predicciones)
mse     = mean_squared_error(y_test, predicciones)
mae_usd = mae * (y_max - y_min)

print("================================================")
print("MÉTRICAS — Regresión Lineal Múltiple")
print("================================================")
print(f"R²:        {r2:.4f}  → explica el {r2*100:.1f}% de la variación")
print(f"MAE real:  ${mae_usd:>12,.0f} USD en promedio")
print(f"MSE:       {mse:.4f}")
print(f"\n→ El {(1-r2)*100:.1f}% de la variación sigue sin explicarse.")

In [ ]:
# ============================================================
# 10. GRÁFICAS DE DIAGNÓSTICO
# ============================================================
# Gráfica 1 — Real vs Predicción:
#   Si los puntos siguieran la línea roja → predicción perfecta.
#   Puntos dispersos lejos de la línea = el modelo falla.
#
# Gráfica 2 — Residuos (error = real - predicción):
#   Residuos aleatorios alrededor de 0 → modelo OK.
#   Patrón visible (curva, abanico) → hay relaciones no lineales
#   que el modelo lineal no está capturando.

y_test_r  = y_test        * (y_max - y_min) + y_min
pred_r    = predicciones  * (y_max - y_min) + y_min
residuos  = (y_test - predicciones) * (y_max - y_min)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Real vs Predicción
axes[0].scatter(y_test_r, pred_r, alpha=0.3, s=12, color='steelblue')
lim = [y_test_r.min(), y_test_r.max()]
axes[0].plot(lim, lim, 'r--', linewidth=1.5, label='Predicción perfecta')
axes[0].set_title('Ventas reales vs Predicciones', fontsize=13)
axes[0].set_xlabel('Ventas reales (USD)')
axes[0].set_ylabel('Ventas predichas (USD)')
axes[0].legend()
axes[0].grid(True, alpha=0.25)

# Residuos
axes[1].scatter(pred_r, residuos, alpha=0.3, s=12, color='orangered')
axes[1].axhline(0, color='black', linewidth=1.2, linestyle='--')
axes[1].set_title('Residuos vs Predicciones\n(si hay patrón → modelo insuficiente)', fontsize=13)
axes[1].set_xlabel('Ventas predichas (USD)')
axes[1].set_ylabel('Error (USD)')
axes[1].grid(True, alpha=0.25)

plt.tight_layout()
plt.show()

print(f"Residuos — desviación estándar: ${residuos.std():,.0f} USD")
print(f"Residuos — peor error:          ${residuos.abs().max():,.0f} USD")
print("\n→ La gráfica de residuos muestra un patrón en abanico:")
print("  el modelo subestima sistemáticamente las tiendas con ventas altas.")
print("  Eso indica que hay relaciones NO lineales en los datos.")
print("  → La Regresión Polinomial puede capturar esas curvas.")

In [ ]:
# ============================================================
# 11. PREDICCIÓN — nueva semana
# ============================================================
nuevo = {
    'Store': 1, 'Holiday_Flag': 0, 'Temperature': 55.0,
    'Fuel_Price': 3.1, 'CPI': 211.0, 'Unemployment': 8.1,
    'month': 3, 'week': 10
}

X_nuevo      = np.array([[nuevo[f] for f in features]])
X_nuevo_norm = (X_nuevo - X_min) / (X_max - X_min)
pred_norm    = modelo.predict(X_nuevo_norm)
pred_real    = (pred_norm[0] * (y_max - y_min)) + y_min

historico = df[df['Store']==nuevo['Store']]['Weekly_Sales'].mean()

print("PREDICCIÓN — Tienda 1, semana 10, marzo")
print(f"  Modelo RLM:             ${pred_real:>12,.2f} USD")
print(f"  Promedio histórico:     ${historico:>12,.2f} USD")

---
## 📌 Conclusión

| Métrica | Resultado | Interpretación |
|---|---|---|
| **R²** | 0.1542 | Solo explica el 15% de la variación |
| **MAE real** | ~$433,000 USD | Error promedio muy alto |
| **Residuos** | Patrón visible | Hay relaciones no lineales |

La Regresión Lineal Múltiple **es un punto de partida válido** y funciona bien  
cuando los datos tienen relaciones aproximadamente lineales (como el seguro médico).  

En este dataset, las ventas de Walmart tienen comportamientos no lineales  
(pico navideño, diferencias enormes entre tiendas) que una línea recta no puede capturar.  
El siguiente paso natural es la **Regresión Polinomial**.